# Chapter 5 — Validation, limitations, and the contemporary external-generator challenge

Two distinct jobs, deliberately kept apart:

1. **Sections 1-4** validate what has already been run. Everything is computed from the
   saved artefacts; no check below is asserted from a config flag when it can be verified
   from recorded sample IDs or digests instead.
2. **Section 5** reports the external challenge, which has now been **executed** via the
   non-recommended Route B. It is an evaluation-only study on 200 images with an
   **unidentifiable** generator, held entirely apart from the Chapter 4 tables.

**Nothing in this notebook trains a model or generates an image.**

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.evaluation import dissertation as D

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

OUTPUT_ROOT = REPO_ROOT / "outputs"
ctx = D.load_context(OUTPUT_ROOT)
print(f"repository root : {REPO_ROOT}")
print(f"reportable runs : {len(ctx.records)}")

repository root : /Users/liv.emms/MSc_Project/MSc_Project
reportable runs : 8


## 1. Reproducibility validation

### 1.1 Run provenance

Every run's identity, status, head type and starting checkpoint. The `inclusion` column
records why an excluded run was excluded, which the chapter needs to state rather than
quietly omit.

In [2]:
inventory = pd.DataFrame(D.experiment_inventory(ctx))
inventory[["run_id", "protocol", "status", "inclusion", "held_out_generator",
           "head_type", "seed", "torch_version", "manifest_sha256"]]

,run_id,protocol,status,inclusion,held_out_generator,head_type,seed,torch_version,manifest_sha256
0,ablation-20260809T194436211721Z-fc1a22d8e3-5389,ablation,incomplete,excluded: status incomplete,biggan,linear,42,None,NaN
1,ablation-20260816T162529520026Z-fc1a22d8e3-f09d,ablation,incomplete,excluded: status incomplete,biggan,linear,42,None,NaN
2,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,ablation,completed,included,biggan,linear,42,None,3a5d4aa23f77cebc514c0362da3bfe6aef0139dcb2a663...
3,ablation-20260823T185746201181Z-fe8a762962-59b7,ablation,completed,included,vqdm,linear,42,None,3a5d4aa23f77cebc514c0362da3bfe6aef0139dcb2a663...
4,baseline-20260809T150824861195Z-4c050175ac-2692,baseline,completed,included,NaN,linear,42,None,NaN
5,fine_tuning-20260808T144640138049Z-6ec6cf50a2-...,fine_tuning,completed,excluded: synthetic smoke run,biggan,linear,42,None,NaN
6,fine_tuning-20260808T171227711418Z-314210675e-...,fine_tuning,completed,included,biggan,linear,42,None,3a5d4aa23f77cebc514c0362da3bfe6aef0139dcb2a663...
7,fine_tuning-20260816T204608740055Z-462647a0cf-...,fine_tuning,completed,included,vqdm,linear,42,None,3a5d4aa23f77cebc514c0362da3bfe6aef0139dcb2a663...
8,unseen_generator-20260808T144550443491Z-b386ea...,unseen_generator,completed,excluded: synthetic smoke run,biggan,linear,42,None,NaN
9,unseen_generator-20260808T150151025524Z-c74c3e...,unseen_generator,failed,excluded: status failed,biggan,linear,42,None,NaN


### 1.2 The exact-reproduction check

The VQDM depth ablation refitted four cells that the standalone VQDM recovery run had
already fitted: same starting checkpoint, same subset seed, same nested subsets, same
byte-identical final test set. The head-only cells therefore act as an end-to-end repeat
of the whole data path — subset construction, checkpoint reload, training, evaluation —
rather than a cached value being read twice.

Agreement here is the reproducibility claim. Disagreement would be a finding.

In [3]:
repro = pd.DataFrame(D.reproduction_checks(ctx))
repro[["held_out_generator", "cell", "metric", "ablation_value", "recovery_value",
       "difference", "exact"]].reset_index(drop=True)

,held_out_generator,cell,metric,ablation_value,recovery_value,difference,exact
0,biggan,none_p00,roc_auc,0.9284,0.9284,0.0000,True
1,biggan,none_p00,average_precision,0.9243,0.9243,0.0000,True
2,biggan,none_p00,f1,0.8462,0.8462,0.0000,True
3,biggan,head_only_p05,roc_auc,0.9843,0.9843,0.0000,True
4,biggan,head_only_p05,average_precision,0.9833,0.9833,0.0000,True
5,biggan,head_only_p05,f1,0.8519,0.8519,0.0000,True
6,biggan,head_only_p10,roc_auc,0.9938,0.9938,0.0000,True
7,biggan,head_only_p10,average_precision,0.9935,0.9935,0.0000,True
8,biggan,head_only_p10,f1,0.8952,0.8952,0.0000,True
9,biggan,head_only_p20,roc_auc,0.9960,0.9960,0.0000,True


In [4]:
summary = (
    repro.groupby("held_out_generator")
    .agg(comparisons=("exact", "size"),
         exact=("exact", "sum"),
         agrees_to_4dp=("agrees_to_4dp", "sum"),
         largest_absolute_difference=("difference", lambda s: s.abs().max()))
    .reset_index()
)
summary

,held_out_generator,comparisons,exact,agrees_to_4dp,largest_absolute_difference
0,biggan,15,15,15,0.0000
1,vqdm,15,15,15,0.0000


### 1.3 Checkpoint provenance and head separation

Held-out generator alone does **not** identify a run in this project: VQDM is held out
under both a linear and a cosine head. The in-distribution ceiling each recovery curve is
measured against is therefore resolved from the `starting_checkpoint` the adaptation cell
recorded, which names exactly one unseen run.

The check below confirms each ablation's checkpoint resolves to a discovered unseen run
with the *matching* held-out generator, and reports that run's head type.

In [5]:
provenance = pd.DataFrame(D.provenance_checks(ctx))
provenance[["run_id", "held_out_generator", "starting_checkpoint_owner_run",
            "owner_run_discovered", "owner_run_head_type", "owner_run_held_out",
            "compatibility_warnings"]]

,run_id,held_out_generator,starting_checkpoint_owner_run,owner_run_discovered,owner_run_head_type,owner_run_held_out,compatibility_warnings
0,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,unseen_generator-20260808T151948963300Z-c74c3e...,True,linear,biggan,0
1,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,unseen_generator-20260816T191041040059Z-4c30e1...,True,linear,vqdm,0


In [6]:
# Confirm the resolution actually used checkpoint provenance rather than the weaker
# generator-name fallback, across every recovery row the aggregation layer produced.
recovery_rows = pd.DataFrame(ctx.recovery)
match_counts = (
    recovery_rows.groupby(
        ["held_out_generator", "head_type", "in_distribution_reference_match"], dropna=False
    )
    .size()
    .rename("rows")
    .reset_index()
)
match_counts

,held_out_generator,head_type,in_distribution_reference_match,rows
0,biggan,linear,starting_checkpoint,112
1,biggan,linear,NaN,176
2,vqdm,linear,starting_checkpoint,112
3,vqdm,linear,NaN,176


In [7]:
resolved = recovery_rows[recovery_rows["in_distribution_reference_match"].notna()]
fallback = resolved[resolved["in_distribution_reference_match"] == "held_out_generator"]
print(f"rows with a resolved in-distribution ceiling : {len(resolved)}")
print(f"resolved via starting_checkpoint provenance  : "
      f"{(resolved['in_distribution_reference_match'] == 'starting_checkpoint').sum()}")
print(f"resolved via the weaker generator fallback   : {len(fallback)}")
print()
print("reference run used, per generator and head:")
print(resolved.groupby(["held_out_generator", "head_type"])
      ["in_distribution_reference_run_id"].unique().to_string())

rows with a resolved in-distribution ceiling : 224
resolved via starting_checkpoint provenance  : 224
resolved via the weaker generator fallback   : 0

reference run used, per generator and head:
held_out_generator  head_type
biggan              linear       [unseen_generator-20260808T151948963300Z-c74c3...
vqdm                linear       [unseen_generator-20260816T191041040059Z-4c30e...


**Reading.** No row falls back to generator matching. The VQDM reference resolves to the
*linear* unseen run, which is the arm the recovery curves actually started from — the
error this mechanism exists to prevent.

Rows with no resolved ceiling are correct rather than missing: threshold-dependent
metrics at non-default operating points are deliberately left undefined, and
accuracy/precision/recall have no entry in the saved `generalisation_gap` block, so there
is no prevalence-matched ceiling to quote and none is invented.

### 1.4 Manifest and split checks

Digests recorded by the runs themselves, plus the fixed-test-set composition.

In [8]:
provenance[["held_out_generator", "final_test_sha256", "final_test_size",
            "final_test_prevalence", "final_test_policy", "real_pool_sha256",
            "budget_policy", "budget_policy_violations", "manifest_sha256"]]

,held_out_generator,final_test_sha256,final_test_size,final_test_prevalence,final_test_policy,real_pool_sha256,budget_policy,budget_policy_violations,manifest_sha256
0,biggan,6791012c43e9f645729b5f99db38c4c44d66b76b3ed3ca...,500,0.5000,balanced_50_50_fixed_real_pool,3e80924f63f5035baf86967e7f1d699a213ebc31fa2ecd...,equal_epochs,0,3a5d4aa23f77cebc514c0362da3bfe6aef0139dcb2a663...
1,vqdm,861b69f7d914522e574a34056145d48bbf585233a8516d...,500,0.5000,balanced_50_50_fixed_real_pool,3e80924f63f5035baf86967e7f1d699a213ebc31fa2ecd...,equal_epochs,0,3a5d4aa23f77cebc514c0362da3bfe6aef0139dcb2a663...


In [9]:
# Are the two ablations measured on the same test set as each other? They should NOT be:
# each generator has its own held-out test set.
for _, row in provenance.iterrows():
    print(f"{row['held_out_generator']:8s} final_test_sha256 {row['final_test_sha256']}")
print()
print("distinct final test sets:", provenance["final_test_sha256"].nunique(),
      "for", len(provenance), "ablations (expected: one per generator)")
print("shared authentic pool  :", provenance["real_pool_sha256"].nunique(),
      "distinct real_pool_sha256 (expected: 1, the same fixed authentic pool)")

biggan   final_test_sha256 6791012c43e9f645729b5f99db38c4c44d66b76b3ed3ca88fd5728ece53402ec
vqdm     final_test_sha256 861b69f7d914522e574a34056145d48bbf585233a8516da6987da82c3b06b2f0

distinct final test sets: 2 for 2 ablations (expected: one per generator)
shared authentic pool  : 1 distinct real_pool_sha256 (expected: 1, the same fixed authentic pool)


## 2. Leakage validation

### 2.1 Adaptation-to-test sample-ID overlap

The check that matters most. Any non-zero intersection would mean a recovery number was
measured on images the cell had been fitted on. Computed by intersecting the saved
adaptation sample IDs against the saved final-test sample IDs, not inferred from the
splitting code.

In [10]:
overlap = pd.DataFrame(D.adaptation_test_overlap(ctx))
overlap

,run_id,held_out_generator,subset_seed,budget,adaptation_ids,final_test_ids,overlapping_ids,clean
0,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.05,800,500,0,True
1,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.10,1600,500,0,True
2,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.20,3200,500,0,True
3,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.50,8000,500,0,True
4,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.05,800,500,0,True
5,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.10,1600,500,0,True
6,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.20,3200,500,0,True
7,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.50,8000,500,0,True


In [11]:
print(f"total overlapping sample IDs across all budgets and runs: "
      f"{overlap['overlapping_ids'].sum()}")
print(f"combinations checked: {len(overlap)}, all clean: {overlap['clean'].all()}")

total overlapping sample IDs across all budgets and runs: 0
combinations checked: 8, all clean: True


### 2.2 Nested adaptation subsets

Nesting is what makes the recovery curve a curve rather than five unrelated fits. Verified
from the saved sample IDs: each larger budget must be a strict superset of the smaller.
`train_validation_overlap` must be zero, or a cell would be validating on its own
training images.

In [12]:
nested = pd.DataFrame(D.nested_subset_checks(ctx))
nested[["run_id", "held_out_generator", "subset_seed", "budget", "unique_sample_ids",
        "declared_consumed", "ids_match_declared_count", "train_validation_overlap",
        "previous_budget", "is_superset_of_previous"]]

,run_id,held_out_generator,subset_seed,budget,unique_sample_ids,declared_consumed,ids_match_declared_count,train_validation_overlap,previous_budget,is_superset_of_previous
0,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.05,800,800,True,0,NaN,None
1,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.10,1600,1600,True,0,0.05,True
2,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.20,3200,3200,True,0,0.10,True
3,ablation-20260816T163436467712Z-fc1a22d8e3-4f83,biggan,42,0.50,8000,8000,True,0,0.20,True
4,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.05,800,800,True,0,NaN,None
5,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.10,1600,1600,True,0,0.05,True
6,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.20,3200,3200,True,0,0.10,True
7,ablation-20260823T185746201181Z-fe8a762962-59b7,vqdm,42,0.50,8000,8000,True,0,0.20,True
8,fine_tuning-20260808T171227711418Z-314210675e-...,biggan,42,0.05,800,800,True,0,NaN,None
9,fine_tuning-20260808T171227711418Z-314210675e-...,biggan,42,0.10,1600,1600,True,0,0.05,True


In [13]:
checked = nested[nested["is_superset_of_previous"].notna()]
print(f"nesting checks: {len(checked)}, all supersets: {bool(checked['is_superset_of_previous'].all())}")
print(f"ID counts match declared consumption: {bool(nested['ids_match_declared_count'].all())}")
print(f"train/validation overlap total: {nested['train_validation_overlap'].sum()}")

nesting checks: 12, all supersets: True
ID counts match declared consumption: True
train/validation overlap total: 0


### 2.3 Source-group overlap and content-hash deduplication

These are dataset-construction guarantees, recorded in the manifest audit at build time.

Splitting is **grouped**: `create_grouped_splits` partitions by `source_group`, not by
file, so near-duplicate crops of one source image cannot straddle the train/test boundary.
Authentic images were deduplicated on `sha256_of_file_content` before splitting.

In [14]:
dataset = D.dataset_provenance(ctx)
for key in ["dataset", "dataset_source", "is_tiny_genimage_subset", "sample_count",
            "split_counts", "generators", "excluded_official_generators",
            "nature_deduplication_key", "deduplicated_repeated_nature_files",
            "deduplicated_across_official_split_boundary", "seed"]:
    print(f"{key:46s} {dataset.get(key)}")

dataset                                        GenImage
dataset_source                                 tiny_genimage_kaggle_yangsangtai_v1_subset_of_2306.08571
is_tiny_genimage_subset                        True
sample_count                                   34999
split_counts                                   {'test': 3500, 'train': 27999, 'validation': 3500}
generators                                     ['biggan', 'vqdm', 'stable_diffusion_v1_5', 'wukong', 'adm', 'glide', 'midjourney']
excluded_official_generators                   ['stable_diffusion_v1_4']
nature_deduplication_key                       sha256_of_file_content
deduplicated_repeated_nature_files             1
deduplicated_across_official_split_boundary    0
seed                                           42


In [15]:
print("preprocessing policy (why container format cannot predict the class):")
for key, value in (dataset.get("preprocessing") or {}).items():
    print(f"  {key:24s} {value}")

preprocessing policy (why container format cannot predict the class):
  does_not_remove          ['high-frequency content differences from native generation resolution', 'residue asymmetry between already-JPEG reals and never-compressed fakes']
  jpeg_quality             95
  jpeg_subsampling         0
  output_format            JPEG
  output_mode              RGB
  pillow_version           12.3.0
  policy_identity          e22755d3f6b17b27
  removes                  ['container-format class shortcut (all images re-encoded as JPEG)', 'raw input dimension class/generator shortcut (all images same size)']
  resample_filter          BICUBIC
  resize_policy            shortest_side_then_center_crop
  schema_version           1
  strip_metadata           True
  target_size              256


### 2.4 Fixed test-set protection and restart-from-baseline

Two protocol guarantees, both recorded per run:

- The unseen test set is built once per generator and is byte-identical across every
  budget and depth (`final_test_sha256` above).
- Every adaptation cell reloads the *original* starting checkpoint rather than continuing
  from the previous budget's weights (`reload_starting_checkpoint_each_run`), so the
  budgets are independent fits from one common origin rather than a single training run
  sampled at five points.

In [16]:
import yaml

for record in ctx.ablation_runs():
    resolved = yaml.safe_load((OUTPUT_ROOT / record.run_id / "resolved_config.yaml").read_text())
    fine_tuning = resolved.get("fine_tuning") or {}
    print(f"{ctx.held_out_of(record.run_id):8s} "
          f"reload_starting_checkpoint_each_run={fine_tuning.get('reload_starting_checkpoint_each_run')} "
          f"nested_subsets={fine_tuning.get('nested_subsets')} "
          f"final_test_split={fine_tuning.get('final_test_split_name')}")

biggan   reload_starting_checkpoint_each_run=True nested_subsets=True final_test_split=unseen_test
vqdm     reload_starting_checkpoint_each_run=True nested_subsets=True final_test_split=unseen_test


### 2.5 Consolidated validation table

Every check above as one pass/fail table. This is exported as `validation_summary.csv`.

In [17]:
validation = pd.DataFrame(D.validation_summary(ctx))
by_category = (
    validation.groupby("category")
    .agg(checks=("passed", "size"), passed=("passed", "sum"))
    .reset_index()
)
by_category

,category,checks,passed
0,leakage,8,8
1,nesting,12,12
2,provenance,2,2
3,reproduction,2,2
4,test-set protection,2,2
5,training budget,2,2


In [18]:
failures = validation[~validation["passed"]]
if failures.empty:
    print("All validation checks passed.")
else:
    print(f"{len(failures)} FAILING CHECKS:")
    display(failures)

All validation checks passed.


In [19]:
validation

,category,check,result,passed,detail
0,reproduction,biggan: ablation head-only cells vs standalone...,15/15 exact,True,"same starting checkpoint, subset seed and fina..."
1,reproduction,vqdm: ablation head-only cells vs standalone r...,15/15 exact,True,"same starting checkpoint, subset seed and fina..."
2,provenance,biggan: starting checkpoint resolves to a disc...,unseen_generator-20260808T151948963300Z-c74c3e...,True,"owner head_type=linear, compatibility warnings=0"
3,test-set protection,biggan: final test set identity,6791012c43e9f645...,True,"policy=balanced_50_50_fixed_real_pool, n=500, ..."
4,training budget,biggan: equal-epoch budget policy honoured,0 violations,True,equal_epochs
5,provenance,vqdm: starting checkpoint resolves to a discov...,unseen_generator-20260816T191041040059Z-4c30e1...,True,"owner head_type=linear, compatibility warnings=0"
6,test-set protection,vqdm: final test set identity,861b69f7d914522e...,True,"policy=balanced_50_50_fixed_real_pool, n=500, ..."
7,training budget,vqdm: equal-epoch budget policy honoured,0 violations,True,equal_epochs
8,leakage,biggan 0.05: adaptation vs final test sample-I...,0 overlapping of 800,True,final test holds 500 sample IDs
9,leakage,biggan 0.10: adaptation vs final test sample-I...,0 overlapping of 1600,True,final test holds 500 sample IDs


## 3. Experimental limitations

Computed from the runs rather than recited from a template: seed counts, test-set size,
saturation counts, generator coverage and excluded runs are all read from what was
actually executed.

In [20]:
limitations = pd.DataFrame(D.limitations(ctx))
for _, row in limitations.iterrows():
    print("=" * 78)
    print(row["limitation"])
    print("=" * 78)
    print(f"  evidence   : {row['evidence']}")
    print(f"  consequence: {row['consequence']}")
    print()

Single subset seed and single training seed
  evidence   : subset seeds run: [42]; one training seed (42)
  consequence: No variance estimate exists. Every reported difference is a difference between two individual fits, so no confidence interval, standard error or significance test can be computed, and none is reported.

Small fixed evaluation set
  evidence   : unseen test support: 500 samples, balanced 50/50
  consequence: At n=500 the coarsest resolvable change in a count-based metric is one sample, so small metric differences correspond to a handful of images.

Ceiling effects on the easier generator
  evidence   : 11 of 26 ablation cells reach ROC-AUC >= 0.99
  consequence: Where the 0% baseline is already near the ceiling, depth and budget cannot be separated from each other because there is nothing left to recover. This is why the depth claim rests on VQDM rather than BigGAN.

Generator coverage
  evidence   : 2 of 7 generators held out with a depth ablation (biggan, vqdm); ben

### 3.1 The internal / external boundary

This is the limitation the external challenge in section 5 exists to address, and it is
worth stating precisely.

Everything measured in Chapter 4 is generalisation to a generator **held out of one
benchmark, assembled at one time, from one source**. That is a real and controlled
measurement of cross-generator generalisation. It is *not* a measurement of how the
detector behaves against generators that did not exist when the benchmark was built.

Section 5 now adds one measurement from outside that benchmark: 200 images, 100 of them
produced in 2026 through an assistant-mediated hosted image-generation tool. It is a
directional probe at a tenth of the internal test's statistical resolution, with a
generator that cannot be named, and it is reported separately from every Chapter 4
number.

In [21]:
generators = sorted(
    {str(r["held_out_generator"]) for r in D.recovery_table(ctx) if r["held_out_generator"]}
)
all_known = sorted(set(dataset.get("generators") or []))

# The Chapter 4 context deliberately does not discover external_challenge runs, so the
# external set is counted here from its own manifest rather than from ctx.
external_manifest = REPO_ROOT / "data/manifests/external_challenge_v1.csv"
external_count = 0
if external_manifest.is_file():
    with external_manifest.open(encoding="utf-8", newline="") as handle:
        external_count = sum(1 for _ in csv.DictReader(handle))

print(f"generators in the benchmark        : {all_known}")
print(f"generators held out with a full    : {generators}")
print(f"  depth ablation")
print(f"images evaluated from outside      : {external_count} "
      f"(external challenge, reported in section 5 only)")
print(f"  the benchmark")

generators in the benchmark        : ['adm', 'biggan', 'glide', 'midjourney', 'stable_diffusion_v1_5', 'vqdm', 'wukong']
generators held out with a full    : ['biggan', 'vqdm']
  depth ablation
images evaluated from outside      : 200 (external challenge, reported in section 5 only)
  the benchmark


## 4. Sensitivity and robustness checks already present

Reusing saved data only. No repeated run is invented, and where a robustness question
cannot be answered from what exists, that is stated rather than approximated.

In [22]:
# What robustness evidence DOES exist, and what does not.
checks = [
    ("Repeat of the same cell in an independent run",
     "YES", f"{int(repro['exact'].sum())}/{len(repro)} metric comparisons exact "
            "(ablation head-only vs standalone recovery run)"),
    ("Multiple subset seeds",
     "NO", "the completed ablations ran subset seed 42 only; an earlier smoke run used "
           "42 and 123 but is excluded as synthetic"),
    ("Multiple training seeds",
     "NO", "training seed 42 only"),
    ("Two classifier heads on the same held-out generator",
     "PARTIAL", "linear and cosine both run for VQDM at 0% adaptation only; no depth "
                "ablation exists for the cosine head"),
    ("Two held-out generators under an identical protocol",
     "YES", "biggan and vqdm, same depths, budgets, subsets and test-set policy"),
    ("Multiple operating points per cell",
     "YES", "default 0.5, adaptation-selected, baseline-unchanged saved for every cell"),
    ("Sensitivity to per-depth learning rate",
     "NO", "rates were probed once on biggan at the 5% budget and carried over to vqdm "
           "unchanged; not re-probed per generator"),
]
pd.DataFrame(checks, columns=["robustness question", "available", "evidence"])

,robustness question,available,evidence
0,Repeat of the same cell in an independent run,YES,30/30 metric comparisons exact (ablation head-...
1,Multiple subset seeds,NO,the completed ablations ran subset seed 42 onl...
2,Multiple training seeds,NO,training seed 42 only
3,Two classifier heads on the same held-out gene...,PARTIAL,linear and cosine both run for VQDM at 0% adap...
4,Two held-out generators under an identical pro...,YES,"biggan and vqdm, same depths, budgets, subsets..."
5,Multiple operating points per cell,YES,"default 0.5, adaptation-selected, baseline-unc..."
6,Sensitivity to per-depth learning rate,NO,rates were probed once on biggan at the 5% bud...


In [23]:
# The one genuine repeated measurement in the project, quantified.
print("Largest absolute disagreement between the two independent fits of the same cell:")
print(f"  {repro['difference'].abs().max():.10f}")
print()
print("This is an exact-agreement result, not a variance estimate: it shows the pipeline")
print("is deterministic given identical inputs. It says nothing about how much a metric")
print("would move under a different subset seed or training seed, which was never run.")

Largest absolute disagreement between the two independent fits of the same cell:
  0.0000000000

This is an exact-agreement result, not a variance estimate: it shows the pipeline
is deterministic given identical inputs. It says nothing about how much a metric
would move under a different subset seed or training seed, which was never run.


## 5. Contemporary external-generator challenge

**Status: EXECUTED, via Route B — the non-recommended route.** The images were generated
outside this repository through an assistant's hosted image-generation tool, which selects
its own underlying image model and does not report which. The generator is therefore
recorded as `astra_mediated_unidentified` and **no architectural claim attaches to any
number in this section**.

Sections 5.1-5.6 keep the design record: what was checked before designing the study, the
two routes and what each can claim, the protocol, sample size, provenance risks and
implementation steps. Section 5.8 reports what was actually run and what it does and does
not support.

### 5.1 What the generation route actually is

The brief referred to "GPT-6 Astra". Before designing anything around it, the actual
generation route was checked against OpenAI's published documentation, because whether
Astra is an image generator determines what this experiment is even allowed to claim.

**What the documentation says:**

| Fact | Source |
|---|---|
| Model id `gpt-6-astra`, released 2026-09-03 | OpenAI model page |
| Input modalities: text, image. **Output modality: text** | OpenAI API model reference |
| Image generation is a hosted **tool** (`image_generation`) the model can call, listed alongside computer use, code interpreter, web search | OpenAI API model reference |
| The tool selects the image model itself from the GPT Image family (`gpt-image-2`, `gpt-image-1.5`, `gpt-image-1`, `gpt-image-1-mini`) | OpenAI image-generation guide |
| The caller **cannot pin** which GPT Image model is used, and the tool-call result **does not report** which was used | OpenAI image-generation guide |
| The direct Images API **does** accept an explicit model id | OpenAI image-generation guide |

**Conclusion.** GPT-6 Astra is not an image-generation architecture. Calling the study
"the Astra generator" would be a provenance error, not a naming preference. An
Astra-routed image cannot be attributed to any named generator at all. This is exactly
what happened: the shipped provenance CSV records `model_identifier` as `unavailable` for
all 100 images.

In [24]:
# The environment as it actually stands, so the notebook does not imply a route it lacks.
import importlib.util
import os

print("openai SDK installed     :", importlib.util.find_spec("openai") is not None)
print("OPENAI_API_KEY configured:", bool(os.environ.get("OPENAI_API_KEY")))
print()
print("Neither is present. Route A (direct Images API with a pinned model id) was")
print("therefore never reachable from this environment, which is why the executed study")
print("is Route B and why its generator cannot be attributed to a named architecture.")

openai SDK installed     : False
OPENAI_API_KEY configured: False

Neither is present. Route A (direct Images API with a pinned model id) was
therefore never reachable from this environment, which is why the executed study
is Route B and why its generator cannot be attributed to a named architecture.


### 5.2 Two defensible designs, answering different questions

| | **Route A (recommended)** | **Route B** |
|---|---|---|
| Call | Images API, pinned model id | Astra + `image_generation` tool |
| Question answered | does the detector generalise to a *named* contemporary generator? | does it generalise to what a contemporary *assistant* produces? |
| Generator identity | recorded exactly | unidentifiable |
| Dissertation label | e.g. "gpt-image-2 challenge" | "Contemporary OpenAI/Astra-mediated image-generation challenge" |
| Main risk | none material | no architectural claim is possible; silent model rotation between batches |

**Recommendation: Route A.** It answers the question the dissertation actually asks —
whether a detector trained on a 2023-era benchmark holds up against a generator released
after it — and it does so with attributable provenance.

If Route B is run anyway for its framing, then the underlying generator must be recorded
as *unidentified*, the study must carry the Astra-mediated name, and no claim about any
generator architecture may be attached to the result.

### 5.3 Protocol

Ordered so that no external image can influence development, training or model selection.

1. **Freeze first.** Nominate the checkpoint and record its sha256 *before* any external
   image exists. The frozen detector is evaluated first and once.
2. **Pre-register the prompt set.** Derive prompts from the ImageNet class vocabulary the
   internal benchmark already uses, so subject matter is not confounded with generator
   identity. Save prompts verbatim with stable ids.
3. **Generate with provenance.** Record every field listed in section 5.6 including the
   raw API response metadata. Hash every image on receipt.
4. **Select authentic comparators by protocol, not by eye.** Draw from the held-out
   authentic pool, class-balanced, matched on resolution, format, compression and colour
   mode, with a recorded seed.
5. **Normalise identically.** Push external images through the existing preprocessing
   policy so container format cannot predict the class — the same guarantee
   `scripts/verify_corrections.py` already enforces internally.
6. **Evaluate once, report separately.** No threshold re-selection, no adaptation, no
   merging into Chapter 4 tables.
7. **Only then, optionally,** run the same limited-data recovery protocol against the
   external set as a second, clearly-separated study.

In [25]:
# The candidate frozen checkpoints, from the saved runs.
candidates = []
for record in ctx.by_type("unseen_generator"):
    digest = (record.artefact_digests or {}).get("best_checkpoint.pt")
    candidates.append({
        "run_id": record.run_id,
        "held_out_generator": ctx.held_out_of(record.run_id),
        "head_type": record.head_type,
        "best_checkpoint_sha256": digest,
    })
pd.DataFrame(candidates)

,run_id,held_out_generator,head_type,best_checkpoint_sha256
0,unseen_generator-20260808T151948963300Z-c74c3e...,biggan,linear,dabf51b08cfce27ab8efacb82e21e0e5e3caf1de317b61...
1,unseen_generator-20260816T191041040059Z-4c30e1...,vqdm,linear,3555e68a3396da4b98713bd29b52a56d5411afd96c8386...
2,unseen_generator-20260816T230229229836Z-e41f75...,vqdm,cosine,222cac1e367f0426f9105096b0b27a6a716ca264fb7bf4...


Any of these can be the frozen detector. The choice recorded in
`configs/external_challenge_v1.yaml` **before** the evaluation ran is the **linear VQDM**
run, because it is the checkpoint the entire depth ablation started from, so an external
result is directly comparable to the internal recovery curves. The other three frozen
checkpoints are also evaluated and also reported, so the headline cannot be a post-hoc
pick of whichever detector scored best.

### 5.4 Sample size, cost and time

| Option | Images/class | Total | Purpose |
|---|---|---|---|
| Pilot | 50 | 100 | provenance smoke test; confirm metadata capture works end to end |
| Minimum reportable | 100 | 200 | directional result only |
| **Recommended** | **250** | **500** | matches the internal unseen-test size exactly, so ROC-AUC resolution is directly comparable |

**Cost cannot be stated from this environment.** There is no credential here and the
per-image price of the pinned image model has not been verified. It must be read from
current pricing immediately before the run and recorded in the manifest. Generating 250
images is a small job; the binding constraint is careful provenance capture, not compute
or spend.

**Time.** Generation is minutes to low hours depending on rate limits. Evaluating a frozen
detector on 500 images is a few minutes on the same CPU-only machine that ran the internal
experiments — the 500-image internal test sets evaluate in well under a minute per cell.

### 5.5 Provenance risks

| Risk | Severity | Mitigation |
|---|---|---|
| Unidentifiable underlying generator (Route B) | **high** | choose Route A; otherwise record as unidentified and make no architectural claim |
| Silent model rotation between batches | medium | record model id and response id per image; generate in one session; hash everything |
| Provider-side post-processing (watermarks, C2PA, re-encoding) detectable as a *format* artefact rather than a generator artefact | **high** | strip metadata in the existing preprocessing step; report what was stripped; verify format non-predictiveness on the external set |
| Prompt-subject confound | medium | derive prompts from the same class vocabulary as the internal benchmark |
| Refusals and content filtering skewing the prompt distribution | medium | record refusals; do not silently resample |
| Authentic comparators drawn from a different era/source than the generated set | medium | draw from the held-out authentic pool; match resolution, format, compression, colour mode |

### 5.6 Exact implementation steps

1. Add `openai` to `requirements.txt`; configure a credential **outside** the repository.
2. Write `configs/external_challenge_v1.yaml` recording route, model id, size, quality,
   format, prompt-set id, and the frozen checkpoint sha256.
3. Write `scripts/generate_external_challenge.py` — generation and hashing **only**,
   writing images plus one provenance record per image. No evaluation in this script.
4. Write `scripts/build_external_manifest.py` — assemble the dataset manifest in the
   existing schema so the standard evaluator can read it unmodified.
5. Extend `scripts/verify_corrections.py` with an external-set check: class balance,
   format non-predictiveness, and zero overlap with any internal split.
6. Evaluate the frozen detector with the existing evaluator, writing to a **separate** run
   directory and a **separate** export directory.
7. Report in Chapter 5 only, applying the naming decision from section 5.1.

### 5.7 Machine-readable manifest

Written to `outputs/report/dissertation_results/external_challenge_manifest.json` by
`scripts/build_dissertation_results.py`. It carries both the design and, once an external
run exists under `outputs/`, an `executed` block with that run's composition, leakage
checks and numbers. Provenance fields that were never established stay `null` by design,
so the manifest cannot be read as claiming something that was not verified.

In [26]:
from scripts.build_dissertation_results import external_challenge_manifest

manifest = external_challenge_manifest(ctx)
print(json.dumps(manifest, indent=2))

{
  "challenge_id": "contemporary_openai_astra_mediated_v1",
  "title": "Contemporary OpenAI/Astra-mediated image-generation challenge",
  "status": "executed",
  "executed": {
    "run_id": "external_challenge-20260908T204005035845Z-fa130b2476-2bce",
    "run_dir": "/Users/liv.emms/MSc_Project/MSc_Project/outputs/external_challenge-20260908T204005035845Z-fa130b2476-2bce",
    "route_used": "astra_mediated_hosted_image_generation_tool",
    "route_option": "astra_responses_api_image_generation_tool (Route B)",
    "generator_recorded_as": "astra_mediated_unidentified",
    "generator_identity_known": false,
    "architectural_claim_permitted": false,
    "evaluation_composition": {
      "authentic_comparator": 100,
      "external_fake": 100,
      "external_set_sha256": "f656570a181548deade7b9bbdf47c351de72bb3c6c83745702e369fa2af93e81",
      "positive_prevalence": 0.5,
      "total": 200
    },
    "sample_size_tier": {
      "images_per_class": 100,
      "recommended_tier_reached"

In [27]:
# Guard: the manifest must agree with what is actually on disk, in either direction.
external_runs = sorted(OUTPUT_ROOT.glob("external_challenge-*"))
if manifest["status"] == "proposed_not_executed":
    assert not external_runs, "an external run exists but the manifest says it does not"
    for item in manifest["not_yet_done"]:
        print(f"  - {item}")
else:
    assert manifest["status"] == "executed"
    executed = manifest["executed"]
    assert executed["generator_identity_known"] is False
    assert executed["architectural_claim_permitted"] is False
    assert all(check["passed"] for check in executed["leakage_checks"])
    print(f"run              : {executed['run_id']}")
    print(f"route            : {executed['route_option']}")
    print(f"generator         : {executed['generator_recorded_as']} (identity unknown)")
    print(f"composition       : {executed['evaluation_composition']}")
    print(f"sample-size tier  : {executed['sample_size_tier']['tiers_reached'][-1]}")
    print(f"exclusions        : {len(executed['exclusions'] or [])}")
    print()
    print("not done:")
    for item in manifest["not_yet_done"]:
        print(f"  - {item}")

run              : external_challenge-20260908T204005035845Z-fa130b2476-2bce
route            : astra_responses_api_image_generation_tool (Route B)
generator         : astra_mediated_unidentified (identity unknown)
composition       : {'authentic_comparator': 100, 'external_fake': 100, 'external_set_sha256': 'f656570a181548deade7b9bbdf47c351de72bb3c6c83745702e369fa2af93e81', 'positive_prevalence': 0.5, 'total': 200}
sample-size tier  : minimum reportable
exclusions        : 0

not done:
  - step 7 of the protocol (limited-data recovery against the external set) was not run: the recovery and ablation cells retained checkpoint digests only, so no adapted detector exists to score without re-training
  - the recommended 250-per-class sample size was not reached; 100 generated images were available, giving the minimum-reportable tier


### 5.8 What was actually run, and how to read it

The external set is class balanced at 100 generated + 100 authentic. The authentic half is
the first 100 entries of the **same** seeded ordering the internal unseen tests use, so it
is a nested subset of the fixed 250-image real pool: the external and internal numbers
share their negatives and differ only in their positives. Both halves pass through the
identical pinned preprocessing policy, so every evaluated image is a 256x256 RGB JPEG q95
with metadata stripped and container format cannot predict the class.

In [28]:
# Every frozen detector on the external set, beside its own saved internal numbers.
external_summary = OUTPUT_ROOT / "report/external_challenge/tab04_external_challenge.csv"
if external_summary.is_file():
    external = pd.read_csv(external_summary)
    display(external[[
        "detector_role", "is_primary", "head_type", "held_out_generator",
        "external_roc_auc", "external_pr_auc", "external_f1",
        "internal_in_distribution_roc_auc", "internal_unseen_roc_auc",
        "external_minus_unseen_roc_auc", "external_minus_in_distribution_roc_auc",
    ]])
else:
    print("no external challenge exports; run scripts.build_external_figures")

,detector_role,is_primary,head_type,held_out_generator,external_roc_auc,external_pr_auc,external_f1,internal_in_distribution_roc_auc,internal_unseen_roc_auc,external_minus_unseen_roc_auc,external_minus_in_distribution_roc_auc
0,primary_vqdm_held_out_linear,True,linear,vqdm,0.9687,0.9691,0.9057,0.9664,0.6746,0.2941,0.0023
1,all_seven_generators_seen_baseline,False,linear,NaN,0.9116,0.9144,0.8358,0.9339,NaN,NaN,-0.0223
2,biggan_held_out_linear,False,linear,biggan,0.9092,0.9130,0.8229,0.9374,0.9284,-0.0192,-0.0282
3,vqdm_held_out_cosine,False,cosine,vqdm,0.9603,0.9619,0.8900,0.9659,0.6921,0.2682,-0.0056


In [29]:
# The confusion counts behind the primary detector's external F1, so the operating-point
# behaviour is visible rather than inferred from the F1 alone.
if external_summary.is_file():
    primary = external[external["is_primary"]].iloc[0]
    print(f"primary detector : {primary['detector_role']}")
    print(f"checkpoint       : {primary['checkpoint_sha256_prefix']}...")
    print(f"support          : {int(primary['external_support'])}")
    print(f"TP {int(primary['external_tp'])}  FN {int(primary['external_fn'])}  "
          f"FP {int(primary['external_fp'])}  TN {int(primary['external_tn'])}")
    print(f"precision {primary['external_precision']:.4f}  "
          f"recall {primary['external_recall']:.4f}  "
          f"accuracy {primary['external_accuracy']:.4f}")

primary detector : primary_vqdm_held_out_linear
checkpoint       : 3555e68a3396da4b...
support          : 200
TP 96  FN 4  FP 16  TN 84
precision 0.8571  recall 0.9600  accuracy 0.9000


**Reading, and the three readings that are not available.**

The external set is **not** harder than the held-out internal generator; it is roughly as
easy as in-distribution data.

1. **Supported.** Degradation on an unseen generator is *generator-specific*, not a
   function of how recent or how capable the generator is. VQDM — a 2023 discrete-diffusion
   model *inside* the benchmark — defeats this detector far more thoroughly than a 2026
   assistant-mediated image tool does.
2. **Not supported.** "The detector generalises to contemporary generators." The generator
   is unidentifiable, n is 100 per class, and it is one prompt distribution from one
   session.
3. **Not supported, and the main threat to reading 1.** The external fakes may be separable
   on grounds other than generator artefacts. They were generated at 1254x1254 and
   downscaled to 256 — a far larger reduction than any internal image undergoes — and the
   prompt set yields a clean-background product-photograph aesthetic the ImageNet-derived
   authentic pool does not share. The pre-registered mitigations (ImageNet class
   vocabulary, identical preprocessing) address subject matter and container format but
   **not** native-resolution high-frequency content or photographic style.

Reading 3 has to appear wherever the external number appears. Closing it would need a
second external batch generated at a native resolution matched to the authentic pool, and
that has not been run.

**Step 7 of the protocol — limited-data recovery against the external set — was not run.**
The recovery and depth-ablation cells retained checkpoint *digests* only, not weights, so
no adapted detector exists to score without re-training. It is recorded as not done rather
than approximated.

## 6. What this chapter can and cannot conclude

Machine-derived group D from the shared findings module, reproduced here because Chapter 5
is where these boundaries have to be stated explicitly.

In [30]:
for item in D.findings(ctx)["D. Claims that CANNOT be supported from these experiments"]:
    print(f"  - {item}")

  - Any claim of statistical significance, confidence interval or error bar. One subset seed and one training seed were run, so no variance was measured.
  - Any claim that these results generalise to generators outside the Tiny GenImage subset. No image from outside that benchmark has been evaluated.
  - Any claim that a given budget is 'sufficient' in deployment. Attainment levels (0.90/0.95/0.98 ROC-AUC) are reporting conveniences chosen after the fact, not pre-registered operational criteria.
  - Any ranking of the linear against the cosine head beyond the single held-out generator where both were run, and beyond the 0% operating point.
  - Any claim about which architectural component carries the generator-specific signal. Trainable-parameter counts were recorded, but no layer-wise attribution study was run.
  - Any claim that full fine-tuning is universally preferable. It wins where headroom exists and is indistinguishable where it does not, and its cost is 87.46M trainable param

The two most likely over-readings, stated plainly:

- **"Full fine-tuning at 5% outperforms head-only at 50%."** Directionally true on every
  metric and both generators, but the VQDM ROC-AUC margin is +0.0014 and the PR-AUC margin
  +0.0057 — both below the 0.02 reliability floor, from a single seed. Write it as
  *equivalence at one tenth the labelling cost*. The F1 margin (+0.0789) and the
  missed-detection counts are the forms that survive.
- **"The detector generalises to modern generators."** The external challenge in section 5
  does **not** establish this, even though it scores well. Its generator is
  unidentifiable, its sample size is 200, and the native-resolution and photographic-style
  confounds of section 5.8 reading 3 are uncontrolled. What it supports is the narrower
  and more interesting claim that degradation is generator-specific rather than a function
  of generator recency.